In [ ]:
# === TRAINING PRESETS ===
# Dataset: 220,000 samples available

PRESET = 1  # ✓ Quick test with enhanced pattern detection

configs = {
    0: {  # Debug - 8k steps (~2 min) - fast iterations to test reward
        'TRAIN_DATA_SIZE': 16_384,
        'N_ENVS': 8,
        'N_STEPS': 256,
        'BATCH_SIZE': 64,
        'NUM_ITERATIONS': 4,
    },
    1: {  # Quick test - 32k steps (~5 min)
        'TRAIN_DATA_SIZE': 32_768,
        'N_ENVS': 8,
        'N_STEPS': 512,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 8,
    },
    2: {  # Standard - 131k steps (~20 min)
        'TRAIN_DATA_SIZE': 65_536,
        'N_ENVS': 8,
        'N_STEPS': 1024,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
    3: {  # Extended - 524k steps (~1.5 hours)
        'TRAIN_DATA_SIZE': 131_072,
        'N_ENVS': 8,
        'N_STEPS': 2048,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
    4: {  # Max - 1.04M steps (~3 hours)
        'TRAIN_DATA_SIZE': 220_000,
        'N_ENVS': 8,
        'N_STEPS': 4096,
        'BATCH_SIZE': 128,
        'NUM_ITERATIONS': 16,
    },
}

# Load config
cfg = configs[PRESET]
TRAIN_DATA_SIZE = cfg['TRAIN_DATA_SIZE']
N_ENVS = cfg['N_ENVS']
N_STEPS = cfg['N_STEPS']
NUM_ITERATIONS = cfg['NUM_ITERATIONS']
BATCH_SIZE = cfg['BATCH_SIZE']

# Fixed params
LOOKBACK_WINDOW = 288
HIDDEN_DIM = 256
POLICY_LAYERS = [512, 256, 128]
VALUE_LAYERS = [256, 128]
LEARNING_RATE_START = 3e-4
LEARNING_RATE_DECAY = 0.05   # ✓ Reduced from 0.3 (decay slower)
N_EPOCHS = 10
ENT_COEF = 0.5  # ✓ INCREASED: Force more exploration (was 0.3)

DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_SAVE_PATH = "trading_bot"  # ✓ NEW: Different name for pattern model
VECNORM_SAVE_PATH = "vecnormalizepkl"

# Calculated
STEPS_PER_ITERATION = N_ENVS * N_STEPS
TOTAL_TIMESTEPS = STEPS_PER_ITERATION * NUM_ITERATIONS

print(f"Preset {PRESET}: {TOTAL_TIMESTEPS:,} steps | {NUM_ITERATIONS} iters | {N_ENVS} envs | {TRAIN_DATA_SIZE:,} samples")

Preset 1: 32,768 steps | 8 iters | 8 envs | 32,768 samples


In [2]:
import warnings
warnings.filterwarnings('ignore', message='enable_nested_tensor is True')

from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import torch
import torch.nn as nn
import pandas as pd
from environments.simple_trading_env import SimpleTradingEnv
from environments.trading_enhanced_extractor import TradingEnhancedExtractor  # ✓ NEW: Enhanced with patterns

# Load data
df = pd.read_pickle(DATA_PATH)
train_data = df.iloc[0:TRAIN_DATA_SIZE].reset_index(drop=True)
print(f"Loaded {len(df):,} rows | Training on {len(train_data):,} samples")

# Setup policy with ENHANCED pattern recognition
policy_kwargs = dict(
    features_extractor_class=TradingEnhancedExtractor,  # ✓ NEW: Enhanced extractor
    features_extractor_kwargs=dict(hidden_dim=HIDDEN_DIM),
    net_arch=dict(pi=POLICY_LAYERS, vf=VALUE_LAYERS),
    activation_fn=torch.nn.GELU,
    ortho_init=False,  # ✓ Don't use orthogonal init - helps with exploration
)

# Create environments
vec_env = make_vec_env(
    lambda: Monitor(SimpleTradingEnv(train_data, device="cuda", lookback_window=LOOKBACK_WINDOW)), 
    n_envs=N_ENVS
)

# ✓ CRITICAL: Create NEW model with ENHANCED pattern detection
model = PPO(
    "MultiInputPolicy",
    vec_env,
    device="cuda",
    learning_rate=lambda f: LEARNING_RATE_START * (1 - LEARNING_RATE_DECAY * f),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=ENT_COEF,  # Now 0.5 - much higher
    vf_coef=0.5,
    max_grad_norm=0.5,
    target_kl=0.03,
    stats_window_size=288,
    policy_kwargs=policy_kwargs,
    tensorboard_log="./tensorboard_logs/",
    verbose=2
)

print(f"\n✓ Enhanced model with PATTERN RECOGNITION:")
print(f"   📊 Range Detection - Consolidation zones + breakouts")
print(f"   🌊 Elliott Wave - Impulse (12345) + Correction (ABC)")
print(f"   🔄 Reversal Patterns - H&S, Double Top/Bottom, Flags, Pennants")
print(f"   📍 Support/Resistance - Key levels + bounce/break detection")
print(f"   ⚡ SE Blocks - Channel attention for pattern emphasis")
print(f"   🔗 Residual Connections - Better gradient flow\n")

print(f"📊 Watch for:")
print(f"   - entropy_loss should be around -3.5 to -4.5 (not -5.98!)")
print(f"   - ep_rew_mean should improve from -1350 toward 0")
print(f"   - explained_variance should reach 0.3+ by 8k steps")
print(f"   - Bot should now detect bottoms/tops using patterns\n")

# Train
print(f"\nStarting training: {TOTAL_TIMESTEPS:,} steps...")
model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)

# Save
model.save(MODEL_SAVE_PATH)
print(f"\n✓ Saved: {MODEL_SAVE_PATH}")

Loaded 264,323 rows | Training on 32,768 samples


KeyboardInterrupt: 

In [ ]:
# === TEST EXTRACTOR (Run this before training to verify) ===
import torch
from environments.trading_enhanced_extractor import TradingEnhancedExtractor

print("Testing Enhanced Extractor...\n")

# Create a dummy observation matching your environment
dummy_obs = {
    'ohlc_spatial': torch.randn(1, 288, 4),
    'ohlc_temporal': torch.randn(1, 288, 4),
    'rsi_divergence': torch.randn(1, 288, 3),  # RSI + High + Low
    'macd_divergence': torch.randn(1, 288, 5),  # MACD + Signal + Hist + High + Low
    'price_context': torch.randn(1, 288, 12),
    'trend_indicators': torch.randn(1, 288, 10),
    'momentum_oscillators': torch.randn(1, 288, 2),
    'volume_profile': torch.randn(1, 288, 26),
    'trading_sessions': torch.randn(1, 288, 3),
    'account_state': torch.randn(1, 4),
    'position_info': torch.randn(1, 7),
    'performance_metrics': torch.randn(1, 7),
    'daily_vp_bins': torch.randn(1, 288, 54),
    'cumulative_vp_bins': torch.randn(1, 288, 54),
}

# Test extractor
extractor = TradingEnhancedExtractor(
    observation_space=vec_env.observation_space,
    hidden_dim=HIDDEN_DIM
)

try:
    output = extractor(dummy_obs)
    print(f"✅ Extractor works! Output shape: {output.shape}")
    print(f"✅ Expected: [1, {HIDDEN_DIM * 2}] (hidden_dim * 2 for policy+value)")
    print(f"\n✅ Ready to train!")
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\n🔧 Fix the error before training!")

In [ ]:
# === ARCHITECTURE SUMMARY ===
print("\n" + "="*80)
print("ENHANCED EXTRACTOR ARCHITECTURE")
print("="*80)

extractor = model.policy.features_extractor

print(f"\n{'='*80}")
print("PATTERN DETECTION MODULES (NEW!)")
print(f"{'='*80}")
print(f"  1. Range Detection CNN      : 32-dim (consolidation + breakouts)")
print(f"  2. Elliott Wave CNN         : 48-dim (impulse 12345 + correction ABC)")
print(f"  3. Reversal Patterns CNN    : 32-dim (H&S, Double Top/Bottom, Flags)")
print(f"  4. Support/Resistance CNN   : 32-dim (key levels + bounces/breaks)")

print(f"\n{'='*80}")
print("DIVERGENCE DETECTION (ENHANCED WITH PRICE!)")
print(f"{'='*80}")
print(f"  5. RSI Divergence CNN       : 32-dim (RSI + High + Low normalized)")
print(f"  6. MACD Divergence CNN      : 32-dim (MACD + Signal + Hist + High + Low)")

print(f"\n{'='*80}")
print("EXISTING MODULES")
print(f"{'='*80}")
print(f"  7. Spatial OHLC CNN         : 32-dim (candlestick patterns)")
print(f"  8. Temporal OHLC CNN        : 64-dim (price trends, momentum)")
print(f"  9. Price Context Transformer: 32-dim (time, candle structure)")
print(f" 10. Trend Indicators Trans.  : 32-dim (EMAs, momentum)")
print(f" 11. Momentum Oscillators MLP : 16-dim (Stochastic K/D)")
print(f" 12. Volume Profile Trans.    : 32-dim (session VP, naked POCs)")
print(f" 13. Trading Sessions MLP     : 8-dim (ASIA, LONDON, NY)")
print(f" 14. Account State MLP        : 16-dim (balance, margin, equity)")
print(f" 15. Position Info MLP        : 16-dim (position details)")
print(f" 16. Performance Metrics MLP  : 16-dim (win rate, PnL, Sharpe)")
print(f" 17. Daily VP Bins CNN        : 16-dim (intraday volume histogram)")
print(f" 18. Cumulative VP Bins CNN   : 16-dim (all-time accumulation)")

print(f"\n{'='*80}")
print(f"TOTAL FEATURES: 444-dim (was 300-dim before enhancement)")
print(f"OUTPUT: {HIDDEN_DIM * 2}-dim fused features (policy + value)")
print(f"{'='*80}\n")

# Count parameters
total_params = sum(p.numel() for p in extractor.parameters())
trainable_params = sum(p.numel() for p in extractor.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"\n✅ Enhanced with 4 new pattern detection modules!")
print(f"✅ RSI/MACD divergence now includes normalized High/Low price!")